# 🏛️ Fase 2 — Padrões Arquiteturais: MVC, Clean Architecture e Hexagonal
## Separação de Responsabilidades na Prática

---
**Roteiro-Desafio-ES · Fatec SCS · ADS 4º Semestre · 2026**  
**Grupo:** Isaac Gomes  **Data:** 18/05/2026

---

### 🎯 Objetivo desta fase
Implementar a mesma mini-aplicação Java em **3 arquiteturas diferentes** (MVC, Clean Architecture, Hexagonal) e comparar acoplamento, testabilidade e separação de responsabilidades.

### 📚 O que você vai aprender
- Diferenças práticas entre MVC, Clean Architecture e Hexagonal Architecture
- Dependency Inversion Principle (DIP) na prática
- Como a arquitetura impacta testabilidade e manutenibilidade
- Conexão com a arquitetura de agentes FM do artigo CSIRO

### 🔗 Referências
- Robert C. Martin — *Clean Architecture* (2017)
- Alistair Cockburn — *Hexagonal Architecture (Ports and Adapters)* (2005)
- Artigo CSIRO: Fig. 2 — Ecosystem of FM-based agent systems

---


## 🎯 O Domínio: Mini-Sistema de Agente de Tarefas

Vamos construir um sistema simples: um **agente de tarefas** que:
1. Recebe um objetivo do usuário (texto)
2. Gera um plano de ação (lista de passos)
3. Persiste o plano (em memória ou arquivo)
4. Retorna o plano ao usuário

Este domínio mapeia diretamente para o ecossistema do artigo CSIRO: **Passive Goal Creator → Single-Path Plan Generator → Resultado**.

Vamos implementá-lo em 3 arquiteturas.


## 🔹 Passo 1 — Arquitetura MVC (Model-View-Controller)

In [ ]:
// ══ MVC — Model-View-Controller ══
// A separação clássica: Model (dados), View (apresentação), Controller (lógica)

import java.util.*;

// ── MODEL ──
class TaskPlan {
    private String goal;
    private List<String> steps;
    private String createdAt;
    
    public TaskPlan(String goal, List<String> steps) {
        this.goal = goal;
        this.steps = steps;
        this.createdAt = java.time.LocalDateTime.now().toString();
    }
    
    public String getGoal() { return goal; }
    public List<String> getSteps() { return steps; }
    public String getCreatedAt() { return createdAt; }
}

// ── VIEW ──
class TaskPlanView {
    public void displayPlan(TaskPlan plan) {
        System.out.println("╔══════════════════════════════════╗");
        System.out.println("║  📋 PLANO DE AÇÃO (MVC)          ║");
        System.out.println("╠══════════════════════════════════╣");
        System.out.println("  Objetivo: " + plan.getGoal());
        System.out.println("  Criado em: " + plan.getCreatedAt());
        System.out.println("  Passos:");
        int i = 1;
        for (String step : plan.getSteps()) {
            System.out.println("    " + i++ + ". " + step);
        }
        System.out.println("╚══════════════════════════════════╝");
    }
    
    public void displayError(String msg) {
        System.out.println("❌ Erro: " + msg);
    }
}

// ── CONTROLLER ──
class TaskPlanController {
    private TaskPlanView view;
    // Em MVC clássico, o Controller conhece Model e View diretamente
    
    public TaskPlanController(TaskPlanView view) {
        this.view = view;
    }
    
    public void handleRequest(String userGoal) {
        if (userGoal == null || userGoal.isBlank()) {
            view.displayError("Objetivo não pode ser vazio!");
            return;
        }
        
        // Lógica de geração (aqui simplificada)
        List<String> steps = List.of(
            "Analisar o objetivo: '" + userGoal + "'",
            "Decompor em sub-tarefas",
            "Priorizar por dependência",
            "Executar sub-tarefa 1",
            "Validar resultado parcial",
            "Executar sub-tarefas restantes",
            "Consolidar e retornar resultado"
        );
        
        TaskPlan plan = new TaskPlan(userGoal, steps);
        view.displayPlan(plan);
    }
}

// ── Teste MVC ──
TaskPlanView view = new TaskPlanView();
TaskPlanController controller = new TaskPlanController(view);
controller.handleRequest("Criar um sistema de RAG para documentos internos");


## 🔹 Passo 2 — Clean Architecture (Robert C. Martin)

In [ ]:
// ══ CLEAN ARCHITECTURE ══
// Camadas: Entities → Use Cases → Interface Adapters → Frameworks
// Regra de dependência: camadas externas dependem das internas, NUNCA o contrário.

import java.util.*;

// ── CAMADA 1: ENTITIES (núcleo, sem dependências) ──
class PlanEntity {
    private final String goal;
    private final List<String> steps;
    
    public PlanEntity(String goal, List<String> steps) {
        this.goal = goal;
        this.steps = Collections.unmodifiableList(steps);
    }
    
    public String getGoal() { return goal; }
    public List<String> getSteps() { return steps; }
    public int getStepCount() { return steps.size(); }
    
    // Regra de negócio NA ENTIDADE
    public boolean isValid() {
        return goal != null && !goal.isBlank() && !steps.isEmpty();
    }
}

// ── CAMADA 2: USE CASES (regras de aplicação) ──
// Port de saída (interface que o use case EXIGE)
interface PlanRepository {
    void save(PlanEntity plan);
    PlanEntity findByGoal(String goal);
}

// Port de saída para geração
interface PlanGenerator {
    List<String> generateSteps(String goal);
}

// Use Case
class CreatePlanUseCase {
    private final PlanRepository repo;
    private final PlanGenerator generator;
    
    // Dependency Injection — o use case não conhece implementações!
    public CreatePlanUseCase(PlanRepository repo, PlanGenerator generator) {
        this.repo = repo;
        this.generator = generator;
    }
    
    public PlanEntity execute(String goal) {
        List<String> steps = generator.generateSteps(goal);
        PlanEntity plan = new PlanEntity(goal, steps);
        
        if (!plan.isValid()) {
            throw new IllegalArgumentException("Plano inválido para: " + goal);
        }
        
        repo.save(plan);
        return plan;
    }
}

// ── CAMADA 3: INTERFACE ADAPTERS ──
// Implementação do repositório (adapter do port)
class InMemoryPlanRepository implements PlanRepository {
    private Map<String, PlanEntity> store = new HashMap<>();
    
    public void save(PlanEntity plan) {
        store.put(plan.getGoal(), plan);
        System.out.println("  💾 [Repo] Plano salvo em memória");
    }
    
    public PlanEntity findByGoal(String goal) {
        return store.get(goal);
    }
}

// Implementação do gerador (adapter do port)
class SimplePlanGenerator implements PlanGenerator {
    public List<String> generateSteps(String goal) {
        System.out.println("  🧠 [Generator] Gerando plano para: " + goal);
        return List.of(
            "Analisar requisitos de '" + goal + "'",
            "Identificar componentes necessários",
            "Definir interfaces entre componentes",
            "Implementar núcleo (Entities + Use Cases)",
            "Implementar adaptadores (Repos + APIs)",
            "Testes unitários e de integração",
            "Deploy e monitoramento"
        );
    }
}

// ── CAMADA 4: FRAMEWORK (ponto de entrada) ──
// Teste Clean Architecture
System.out.println("╔══════════════════════════════════╗");
System.out.println("║  🏛️ CLEAN ARCHITECTURE            ║");
System.out.println("╠══════════════════════════════════╣");

PlanRepository repo = new InMemoryPlanRepository();
PlanGenerator gen = new SimplePlanGenerator();
CreatePlanUseCase useCase = new CreatePlanUseCase(repo, gen);

PlanEntity result = useCase.execute("Migrar monolito para microsserviços");
System.out.println("\n  📋 Plano gerado:");
System.out.println("  Objetivo: " + result.getGoal());
System.out.println("  Passos (" + result.getStepCount() + "):");
int i = 1;
for (String s : result.getSteps()) System.out.println("    " + i++ + ". " + s);
System.out.println("  Válido? " + result.isValid());
System.out.println("╚══════════════════════════════════╝");


## 🔹 Passo 3 — Hexagonal Architecture (Ports and Adapters)

In [ ]:
// ══ HEXAGONAL ARCHITECTURE (Ports and Adapters) ══
// O DOMÍNIO fica no centro. Ports são interfaces. Adapters são implementações.
// Driving ports (lado esquerdo): quem DIRIGE a app (UI, API, CLI)
// Driven ports (lado direito): quem a app USA (DB, APIs externas)

import java.util.*;

// ── DOMÍNIO (hexágono central) ──
class HexPlan {
    private final String goal;
    private final List<String> steps;
    
    public HexPlan(String goal, List<String> steps) {
        this.goal = goal;
        this.steps = List.copyOf(steps);
    }
    public String getGoal() { return goal; }
    public List<String> getSteps() { return steps; }
}

// ── DRIVING PORT (porta de entrada — interface do serviço) ──
interface PlanService {
    HexPlan createPlan(String goal);
}

// ── DRIVEN PORTS (portas de saída — interfaces para o mundo externo) ──
interface HexPlanStore {
    void persist(HexPlan plan);
}

interface HexStepGenerator {
    List<String> generate(String goal);
}

// ── DOMÍNIO: Implementação do serviço ──
class PlanServiceImpl implements PlanService {
    private final HexPlanStore store;
    private final HexStepGenerator generator;
    
    public PlanServiceImpl(HexPlanStore store, HexStepGenerator gen) {
        this.store = store;
        this.generator = gen;
    }
    
    public HexPlan createPlan(String goal) {
        List<String> steps = generator.generate(goal);
        HexPlan plan = new HexPlan(goal, steps);
        store.persist(plan);
        return plan;
    }
}

// ── DRIVEN ADAPTERS (lado direito — implementações externas) ──
class FileBasedPlanStore implements HexPlanStore {
    public void persist(HexPlan plan) {
        System.out.println("  📁 [FileStore] Plano gravado em /plans/" + 
            plan.getGoal().hashCode() + ".json");
    }
}

class AIStepGenerator implements HexStepGenerator {
    public List<String> generate(String goal) {
        System.out.println("  🤖 [AI Generator] Consultando LLM para: " + goal);
        return List.of(
            "Definir Ports (interfaces de entrada e saída)",
            "Implementar domínio hexagonal central",
            "Criar Driving Adapter (API REST)",
            "Criar Driven Adapter (repositório)",
            "Injetar dependências na composição",
            "Testar domínio isoladamente (sem adapters)",
            "Testar adapters com mocks"
        );
    }
}

// ── DRIVING ADAPTER (lado esquerdo — ponto de entrada CLI) ──
// Teste Hexagonal
System.out.println("╔══════════════════════════════════╗");
System.out.println("║  ⬡ HEXAGONAL ARCHITECTURE         ║");
System.out.println("╠══════════════════════════════════╣");

// Composição — "wiring" dos adapters nos ports
PlanService service = new PlanServiceImpl(
    new FileBasedPlanStore(),    // driven adapter
    new AIStepGenerator()        // driven adapter
);

// Driving adapter simula CLI
HexPlan hexPlan = service.createPlan("Implementar sistema multi-agente");
System.out.println("\n  📋 Plano hexagonal:");
System.out.println("  Objetivo: " + hexPlan.getGoal());
int j = 1;
for (String s : hexPlan.getSteps()) System.out.println("    " + j++ + ". " + s);
System.out.println("╚══════════════════════════════════╝");


## 📊 Passo 4 — Comparação das 3 Arquiteturas

Execute a célula abaixo para ver um resumo comparativo:


In [ ]:
// ══ COMPARAÇÃO ARQUITETURAL ══
System.out.println("╔═══════════════════════════════════════════════════════════════════╗");
System.out.println("║            COMPARAÇÃO: MVC vs CLEAN vs HEXAGONAL                  ║");
System.out.println("╠═══════════════╦═══════════════╦═══════════════╦════════════════════╣");
System.out.println("║ Critério      ║     MVC       ║    Clean      ║    Hexagonal       ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ Camadas       ║ 3 (M-V-C)    ║ 4 (Ent-UC-   ║ 3 (Dom-Ports-      ║");
System.out.println("║               ║               ║  Adpt-Fwk)   ║  Adapters)         ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ Dep. Rule     ║ Bidirecional  ║ Sempre p/     ║ Sempre p/ dentro   ║");
System.out.println("║               ║ (View↔Ctrl)   ║ dentro        ║ (via Ports)        ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ Testabilidade ║ Média (View   ║ Alta (UC      ║ Alta (domínio      ║");
System.out.println("║               ║ acoplada)     ║ isolado)      ║ isolado)           ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ Troca de DB   ║ Difícil       ║ Fácil (troca  ║ Fácil (troca       ║");
System.out.println("║               ║               ║ adapter)      ║ driven adapter)    ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ DIP           ║ Não obrigatório║ Essencial    ║ Essencial          ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ Complexidade  ║ Baixa         ║ Alta          ║ Média-Alta         ║");
System.out.println("╠═══════════════╬═══════════════╬═══════════════╬════════════════════╣");
System.out.println("║ Melhor para   ║ Apps simples, ║ Grandes       ║ Sistemas com       ║");
System.out.println("║               ║ CRUD, web     ║ sistemas,     ║ muitas integrações ║");
System.out.println("║               ║               ║ longa vida    ║ externas           ║");
System.out.println("╚═══════════════╩═══════════════╩═══════════════╩════════════════════╝");

System.out.println("\n💡 O artigo CSIRO (Fig. 2) mostra o ecossistema de agentes FM com:");
System.out.println("   → Driving Adapters: Passive/Proactive Goal Creator (recebem input)");
System.out.println("   → Domínio: Plan Generator, Reflection, Cooperation (lógica central)");
System.out.println("   → Driven Adapters: Tool Registry, Agent Adapter (conexões externas)");
System.out.println("   Isso é essencialmente uma HEXAGONAL ARCHITECTURE para agentes IA!");


## 📝 Avaliação — Fase 2

**Q1.** No padrão MVC, qual componente é responsável pela lógica de negócio?  
( ) Model  ( ) View  (x) Controller  ( ) Todos igualmente

**Q2.** Qual é a "Regra de Dependência" da Clean Architecture?  
( ) Camadas internas dependem das externas  
(X) Camadas externas dependem das internas, nunca o contrário  
( ) Todas as camadas se conhecem mutuamente  
( ) Apenas a camada de framework não tem dependências

**Q3.** Na Hexagonal Architecture, o que são "Ports"?  
( ) Portas de rede para comunicação HTTP  
(X) Interfaces que definem como o domínio se comunica com o mundo externo  
( ) Classes concretas de acesso a banco de dados  
( ) Anotações Java para injeção de dependência

**Q4.** Qual arquitetura facilita mais a troca do banco de dados sem alterar a lógica de negócio?  
( ) MVC (porque o Model é independente)  
(X) Clean Architecture e Hexagonal (porque usam inversão de dependência via interfaces)  
( ) Todas facilitam igualmente  
( ) Nenhuma — trocar DB sempre exige refatoração completa

**Q5.** No artigo CSIRO, o ecossistema de agentes FM (Fig. 2) se assemelha mais a qual arquitetura?  
( ) MVC — porque tem Model (agente), View (usuário) e Controller (plan generator)  
(X) Hexagonal — porque o domínio (agente) se conecta ao mundo externo via adaptadores  
( ) Nenhuma — é uma arquitetura totalmente nova  
( ) Clean Architecture pura — com 4 camadas rígidas

**Q6.** O que é Dependency Inversion Principle (DIP)?  
( ) Classes de alto nível devem depender de classes de baixo nível  
(X) Módulos de alto nível não devem depender de módulos de baixo nível; ambos devem depender de abstrações  
( ) Inversão da ordem de compilação dos módulos  
( ) Uso obrigatório de frameworks de injeção de dependência

**Q7.** Na Clean Architecture, onde fica a regra de negócio mais importante?  
( ) Na camada de Frameworks  (X) Na camada de Entities  ( ) Na camada de Interface Adapters  ( ) No Controller

**Q8.** Na Hexagonal Architecture, qual é a diferença entre Driving Adapter e Driven Adapter?  
( ) Driving é para testes; Driven é para produção  
(X) Driving inicia a interação (ex: CLI, API); Driven é acionado pelo domínio (ex: DB, APIs externas)  
( ) São nomes diferentes para a mesma coisa  
( ) Driving lida com dados de entrada; Driven com dados de saída

**Q9.** Por que o MVC pode ter problemas de testabilidade?  
( ) O Controller é muito complexo para testar  
(X) A View frequentemente acopla lógica de apresentação com acesso a dados  
( ) O Model não pode ser instanciado sem a View  
( ) O Java não suporta testes de Controllers

**Q10.** Qual princípio SOLID é mais exercitado ao implementar Clean/Hexagonal Architecture?  
( ) SRP — Single Responsibility  
( ) OCP — Open/Closed  
( ) LSP — Liskov Substitution  
(X) DIP — Dependency Inversion
